# Gemini Decisions API(gemini-decision-1)란? 글을 생성하지 않고 고르는 API, 출력 0원 가격 구조와 Jev API 비교, Python으로 첫 호출하기

위에서부터 차례로 실행해 보세요.

### 준비: API 키 넣기

Colab 왼쪽의 🔑(보안 비밀)에 `GEMINI_API_KEY` 를 자기 값으로 추가하고 노트북 액세스를 켜 주세요. 키는 노트북에 저장되지 않아요.

In [ ]:
import os
from google.colab import userdata

for name in ['GEMINI_API_KEY']:
    os.environ[name] = userdata.get(name)

### SDK 설치

In [ ]:
!pip install -U google-genai
!pip show google-genai | head -2

### 첫 호출: choice 질문 하나

In [ ]:
%%bash
cat > g12.py <<'E'
import os, time, json
from google import genai
client = genai.Client(http_options={"api_version": "v1beta"})
t=time.time()
r = client.decisions.create(model="gemini-decision-1",
  input="주문한 신발이 아직도 안 왔어요. 환불해 주세요.",
  questions={"topic":{"type":"choice","instructions":"이 문의의 종류는?",
   "criteria":{"refund":"환불 요청","shipping":"배송 문의","other":"기타"}}})
print("elapsed %.0f ms"%((time.time()-t)*1000))
a=r.answers["topic"]
print(a.choice, a.probabilities, a.confidence)
print(r.usage)
E
python3 g12.py

### 질문 4개 묶기, 응답 시간, 비용 계산, 신뢰도 필터

In [ ]:
%%bash
cat > g34.py <<'E'
import time
from google import genai
client = genai.Client(http_options={"api_version": "v1beta"})
text="주문한 신발이 아직도 안 왔어요. 환불해 주세요."
Q={"is_refund":{"type":"noul","instructions":"고객이 환불을 요청하는가?"},
 "is_angry":{"type":"noul","instructions":"고객이 화가 나 있는가?"},
 "topic":{"type":"choice","instructions":"이 문의의 종류는?","criteria":{"refund":"환불 요청","shipping":"배송 문의","other":"기타"}},
 "urgency":{"type":"score","instructions":"긴급도는?","criteria":["여유 있음","보통","즉시 처리 필요"]}}
def call(q):
    t=time.time(); r=client.decisions.create(model="gemini-decision-1",input=text,questions=q)
    return r,(time.time()-t)*1000
call(Q)
for name,q in [("1 question",{"topic":Q["topic"]}),("4 questions",Q)]:
    ms=[]
    for _ in range(3):
        r,m=call(q); ms.append(m)
    print(name,"ms:",[round(x) for x in ms],"input_tokens",r.usage.total_input_tokens,"output",r.usage.total_output_tokens,"cost $%.8f"%(r.usage.total_input_tokens*0.10/1e6))
for k,a in r.answers.items():
    d={f:getattr(a,f) for f in ("noul","choice","score","confidence","probabilities") if getattr(a,f,None) is not None}
    print(k,d)
    if a.confidence<0.8: print("  -> 신뢰도 0.8 미만, 사람에게 넘김")
E
python3 g34.py